Recovered from the kernel's run history (session of 2026-09-30 16:29): the cells exactly as executed, in execution order.

In [124]:
import regex as re

In [125]:
def get_stats(ids)-> dict: # this is just 
    counts = {}
    for i in range(len(ids)):
        for pair in zip(ids[i], ids[i][1:]):
            counts[pair] = counts.get(pair, 0) + 1
    return counts

In [126]:
def merge(ids, pair, idx):

    out = [[] for _ in range (len(ids))]
    for n in range(len(ids)):
        ids_n = ids[n]
        i = 0
        while i < len(ids_n):
            if i < len(ids_n) - 1 and ids_n[i] == pair[0] and ids_n[i + 1] == pair[1]:
                out[n].append(idx)
                i += 2
            else:
                out[n].append(ids_n[i])
                i += 1
    return out

In [127]:
pattern = re.compile("|".join([
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""\p{N}{1,3}""",
    r""" ?[^\s\p{L}\p{N}]+[\r\n/]*""",
    r"""\s*[\r\n]+""",
    r"""\s+(?!\S)""",
    r"""\s+""",
]))

In [ ]:
def train(text, vocab_size, special_tokens,verbose=False): 

    pieces =re.split(re.escape(special_tokens), text) 
    print(f"pieces : {pieces}")
    pre_tokens = [] # This just have no more of the special token 
    for piece in pieces:
        pre_tokens.extend(pattern.findall(piece)) # this create a list of new word

    print(f"pretokens: {pre_tokens}")

    # text = "".join(text)
    #merges: list[tuple[bytes, bytes]]
    ids = list(chunk.encode("utf-8") for chunk in pre_tokens)

    vocab = {i: bytes([i]) for i in range(256)}

    
    print(f"ids {ids}") 
    # 257 : b "cd" 
    merges_list = []
    merges = {} # (97,98) -> 257 

    for idx in range(256, vocab_size - 1 ): # 256 -> 398 
        stats = get_stats(ids)
        if not stats:                                # nothing left to merge
            break
        pair = max(stats, key=lambda p: (stats[p], vocab[p[0]], vocab[p[1]]), )        # tie breaking base on the bytes if want to go ofr the lower use - ? 
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        merges_list.append((vocab[pair[0]], vocab[pair[1]]))
        if verbose:
            print(f"merge {idx - 255}/{vocab_size - 256}: {pair} -> {idx} {vocab[idx]} ({stats[pair]}x)")

        # just do voacg size cuz the loop ends at 399, with vocab sie being 400
    vocab[vocab_size-1] = b'<|endoftext|>' # might need to improve this later this is the lazy way out  
        # vocab[vocab_size -2]
    return vocab, merges_list

In [ ]:
def encode(text):

    # encode from text to remove the special = [ ]
    pieces = re.split(re.escape("<|endoftext|>"), text)

    # another layer to actually use the regex pattenr using extends cuz 
    tokens = [] 
    for piece in pieces:
        tokens.extend(pattern.findall(piece))

    ids =  [token.encode("utf-8") for token in tokens]
    # dont forget that token is a nested loop so we need to loop inside it 
    for idx in range(len(tokens)):
        stats = get_stats(tokens)

        k:v for k,v in tokens in tokens.item() if : 


        top_pair = max(stats, key=lambda p: (stats[p], vocab[p[0]], vocab[p[1]])) # pair we 
        
        for i in range(len(vocab)):
            if (top_pair == vocab[i]):
                idx = vocab.keys()
                ids = merge(ids, top_pair, idx)
                
        # the we compare top_pair with our real top
   
    ids = tokens
    # ids = [] 
    # merge(ids, pair, idx): 

    # merge wil unteract with the thing we would need a top pair 

      

    return ids 



In [ ]:
def decode(ids):
    pass

In [181]:
t = encode("hello i am a cunt <|endoftext|> dick sucker")
print(t)

['hello', ' i', ' am', ' a', ' cunt', ' ', ' dick', ' sucker']


In [ ]:
special_tokens = "<|endoftext|>"

with open("../data/small.txt", "r", encoding="utf-8") as f:
    text = f.read()


vocab, merges = train(text, 400,special_tokens)

# print(merges)

[b'u', b" don't", b' have', b' to', b' be', b' scared', b' of', b' the', b' loud', b' dog', b',', b" I'll", b' protect', b' you', b'".', b' The', b' mole', b' felt', b' so', b' safe', b' with', b' the', b' little', b' girl', b'.', b' She', b' was', b' very', b' kind', b' and', b' the', b' mole', b' soon', b' came', b' to', b' trust', b' her', b'.', b' He', b' leaned', b' against', b' her', b' and', b' she', b' kept', b' him', b' safe', b'.', b' The', b' mole', b' had', b' found', b' his', b' best', b' friend', b'.\n', b'\n', b'Once', b' upon', b' a', b' time', b',', b' in', b' a', b' warm', b' and', b' sunny', b' place', b',', b' there', b' was', b' a', b' big', b' pit', b'.', b' A', b' little', b' boy', b' named', b' Tom', b' liked', b' to', b' play', b' near', b' the', b' pit', b'.', b' One', b' day', b',', b' Tom', b' lost', b' his', b' red', b' ball', b'.', b' He', b' was', b' very', b' sad', b'.\n', b'Tom', b' asked', b' his', b' friend', b',', b' Sam', b',', b' to', b' help', b' 

In [131]:
sorted(vocab) == list(range(400))
print(len(vocab))

400


In [132]:
len(merges)

143

In [134]:
vocab[399]

b'<|endoftext|>'

In [136]:
len(vocab) == 400

True

In [149]:
any(b"<|" in v for k, v in vocab.items() if k != 399)


False

In [165]:
text = "cd cd abab"

vocab, merges=train(text,262,"<|endoftext|>")
print(len(merges))
print(len(vocab))

pieces : ['cd cd abab']
pretokens: ['cd', ' cd', ' abab']
ids [b'cd', b' cd', b' abab']
5
261


In [150]:
sorted(vocab) == list(range(262))

False

In [182]:
any(b"<|" in v for k, v in vocab.items() if k != 262)

True

In [183]:
print(vocab[260])

b'<|endoftext|>'


In [184]:
%pwd

'/Users/cheapanharith/AI/standform-LLMSCRATCH/idontwanttobehomelesstoo/cs336_basics'